In [195]:
import random
import copy
import numpy as np
import json
import vl_convert as vlc
import os

In [196]:
def erase_files(path):

    for filename in os.listdir(path):

        file_path = os.path.join(path, filename)

        try:
            if os.path.isfile(file_path): os.remove(file_path)
        except Exception as e:
            print(f"Error deleting {file_path}: {e}")

In [197]:
def load_list(path):

    with open(path, 'r') as file:
        list = [line.strip() for line in file]

    return list

In [198]:
def render_vega(vega_dict, scale = 1):

    spec = json.dumps(vega_dict, indent = 4)

    try:
        return vlc.vegalite_to_png(vl_spec = spec, scale = scale)
    except:
        print(spec)

def text_to_vega(texts):

    vegas = []
    for text in texts:

        try: vegas.append(json.loads(text.replace("'",'"')))
        except: print("Error in text to vega conversion")

    return vegas

In [199]:
def generate_dataset(n_samples, path):

    erase_files(path)

    metadata = [['file_name','text']]

    # FALTA NOMINAL!!!!

    plots = [{"mark": "bar", "x_type": "nominal"}, 
             {"mark": "line", "x_type": "temporal"},
             {"mark": "circle", "x_type": "quantitative"}]

    var_names = load_list("data/var_names.txt")
    nominals = load_list("data/nominals.txt")

    for i in range(n_samples):

        plot = random.choice(plots)

        x_name = random.choice(var_names)
        y_name = random.choice(var_names)

        x_title = x_name if random.random() < 0.7 else ""
        y_title = y_name if random.random() < 0.7 else ""

        # generation of y values

        y_scale = random.randint(1,1000)
        y_discrete = random.choice([True, False])
        y_values = [np.round(random.random() * y_scale, 2) if not y_discrete else random.randint(1,10) * y_scale for i in range(10)]

        # generation of x values 

        x_values = None

        if plot["x_type"] == "nominal":
            x_values = nominals.copy()
        elif plot["x_type"] == "temporal":
            x_values = list(np.arange(1900,2000,random.randint(1,10)) + random.randint(-100,24))
        elif plot["x_type"] == "quantitative":
            x_scale = random.randint(1,1000)
            x_discrete = random.choice([True, False])
            x_values = [np.round(random.random() * x_scale, 2) if not x_discrete else random.randint(1,10) * x_scale for i in range(10)]

        vega_dict = {
            
            "mark": plot["mark"],
            "encoding": {
                "x": {"field": x_name, "type": plot["x_type"], "title": x_title},
                "y": {"field": y_name, "type": "quantitative", "title": y_title}
            },
            "data": {
                "values": [
                    {x_name: x_values.pop(0), y_name: random.choice(y_values)} for j in range(random.randint(1,10)) 
                ]
            },
        }

        print(vega_dict["data"]["values"])

        image_data = render_vega(vega_dict, scale = 2)

        with open(path + "/" +  str(i) + ".png", "wb") as f: 
            f.write(image_data)

        vega_dict_copy = copy.deepcopy(vega_dict)

        if vega_dict_copy["encoding"]["x"]["title"] == "": vega_dict_copy["encoding"]["x"]["field"] = ""
        if vega_dict_copy["encoding"]["y"]["title"] == "": vega_dict_copy["encoding"]["y"]["field"] = ""

        vega_dict_copy["encoding"]["x"].pop("title")
        vega_dict_copy["encoding"]["y"].pop("title")

        for value_dict in vega_dict_copy["data"]["values"]:

            try:
                value_dict["x"] = value_dict.pop(x_name)
                value_dict["y"] = value_dict.pop(y_name)
            except:
                print("a")

        xml = '"' + str(vega_dict_copy) + '"'

        metadata.append([str(i) + ".png", xml])

    np.savetxt(path + "/metadata.csv", metadata, delimiter = ',', fmt = '% s')

In [200]:
generate_dataset(10, "datasets/visdecode/test")

[{'Education Level': 1849, 'Export Volume': 6174}, {'Education Level': 1851, 'Export Volume': 5488}, {'Education Level': 1853, 'Export Volume': 686}, {'Education Level': 1855, 'Export Volume': 4802}, {'Education Level': 1857, 'Export Volume': 4116}, {'Education Level': 1859, 'Export Volume': 5488}, {'Education Level': 1861, 'Export Volume': 686}, {'Education Level': 1863, 'Export Volume': 5488}]


TypeError: Object of type int64 is not JSON serializable